# UltraGliClass SPEED TEST (throughput / memory probe, NOT a quality run)

40 optimizer steps per config, from the **stage-1 model** on the arm-A pilot data (`data/stage2_pilot`, same prep as the pilot), effective batch 64
(`per_device x accum x 2 GPUs`), no eval (`data.val_files=null`), no intermediate save. Order: **T0** baseline (ckpt, b8) -> **T2** (no ckpt, b4 x accum 8)
-> **T1** (no ckpt, b8) ONLY if T2's peak *reserved* memory is < 8 GB on both ranks -> **T3** (ckpt, b16). Stops on OOM / any failure / NaN.
Per config it reports sec_per_step and peak allocated + reserved memory on rank 0 and rank 1 (train.py ProgressCallback patch of code v4, plus an independent nvidia-smi sampler).
Inputs: code dataset `ultragliclass-code` (v4, with `configs/speedtest_*.yaml`) + output of kernel `ultragliclass-s1`. Settings: GPU T4 x2, Internet on.

In [ ]:
# ---- knobs ----
CONFIGS_ORDER = ["T0_baseline", "T2_nockpt_b4", "T1_nockpt_b8", "T3_ckpt_b16"]   # T1 is gated (see GATE_T1_RESERVED_GB)
GATE_T1_RESERVED_GB = 8.0     # run T1 only if T2 peak reserved (all ranks) is below this
INIT_MODEL    = "auto"
DATA_RECIPE   = "configs/data_stage2_pilot.yaml"
DATA_DIR      = "data/stage2_pilot"
CODE_INPUT    = "auto"
STALL_MINUTES = 10
TOTAL_MAX_HOURS = 0.9         # hard stop: no new config starts after TOTAL_MAX_HOURS - 0.2; each config also gets train.time_budget_hours
CFG_MAX_HOURS = 0.25
EXTRA_SET     = []
import os, time, glob, json, sys, shutil, subprocess, re, signal, threading, queue
T0 = time.time()
LOG_DIR = "/kaggle/working/logs"; os.makedirs(LOG_DIR, exist_ok=True)
NB_LOG = open(os.path.join(LOG_DIR, f"notebook_{time.strftime('%Y%m%d_%H%M%S')}.log"), "a", buffering=1)
def say(*a):
    line = f"[nb {time.strftime('%H:%M:%S')} +{(time.time() - T0) / 60:.1f}m] " + " ".join(str(x) for x in a)
    print(line, flush=True); NB_LOG.write(line + "\n")
def elapsed_h(): return (time.time() - T0) / 3600

def find_code_input():
    hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train.py", recursive=True)]
    hits = [h for h in hits if os.path.exists(os.path.join(h, "configs", "speedtest_T0_baseline.yaml"))]
    hits.sort(key=lambda p: (0 if "ultragliclass-code" in p else 1, len(p)))
    say("code candidates:", hits)
    return hits[0] if hits else None
def find_init_model():
    cands = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/runs/stage1_ultrabert150m/final_model/config.json", recursive=True)] if INIT_MODEL == "auto" else [INIT_MODEL]
    cands.sort(key=len); say("S1 model candidates:", cands)
    return cands[0] if cands else None
if CODE_INPUT == "auto": CODE_INPUT = find_code_input()
INIT_SRC = find_init_model()
say("CODE_INPUT =", CODE_INPUT, "| INIT_SRC =", INIT_SRC)
assert CODE_INPUT and os.path.exists(os.path.join(CODE_INPUT, "configs", "speedtest_T0_baseline.yaml")), "code v4 (configs/speedtest_*.yaml) not found under /kaggle/input"
assert INIT_SRC, "S1 model not found: attach the output of kernel ultragliclass-s1"

In [ ]:
WORK = "/kaggle/working/ultragliclass"
if not os.path.exists(os.path.join(WORK, "train.py")):
    shutil.copytree(CODE_INPUT, WORK, dirs_exist_ok=True, copy_function=shutil.copyfile,
                    ignore=shutil.ignore_patterns("runs", "data", "smoke_logs", "results", "__pycache__"))
subprocess.run(f"chmod -R u+w {WORK}", shell=True)
os.chdir(WORK)
assert "peak_reserved_gb" in open("train.py").read(), "train.py lacks the max_memory_reserved patch (code v4 expected)"
torch_pin = subprocess.run("pip freeze | grep -E '^torch=='", shell=True, capture_output=True, text=True).stdout.strip()
open("/tmp/constraints.txt", "w").write(torch_pin + "\n")
say("torch pin:", torch_pin)
subprocess.run("pip install -q -c /tmp/constraints.txt -r requirements.txt", shell=True, check=True)
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)
import torch, transformers
NGPU = torch.cuda.device_count()
say("torch", torch.__version__, "cuda", torch.cuda.is_available(), "gpus", NGPU, "transformers", transformers.__version__)
assert NGPU >= 2, "this test needs 2 GPUs"
INIT_LOCAL = "/tmp/ugc_init_model"
need = ["config.json", "tokenizer.json", "tokenizer_config.json"]
for f in need: assert os.path.exists(os.path.join(INIT_SRC, f)), f"S1 model dir lacks {f}"
size = sum(os.path.getsize(os.path.join(INIT_SRC, f)) for f in os.listdir(INIT_SRC))
assert size > 200e6, f"S1 model dir is only {size / 1e6:.0f} MB"
shutil.rmtree(INIT_LOCAL, ignore_errors=True); os.makedirs(INIT_LOCAL)
for f in os.listdir(INIT_SRC):
    s = os.path.join(INIT_SRC, f)
    if os.path.isfile(s):
        shutil.copyfile(s, os.path.join(INIT_LOCAL, f)); assert os.path.getsize(os.path.join(INIT_LOCAL, f)) == os.path.getsize(s)
subprocess.run(f"chmod -R u+w {INIT_LOCAL}", shell=True)
say("S1 model copied:", sorted(os.listdir(INIT_LOCAL)), f"{size / 2**20:.0f} MB")

In [ ]:
if not os.path.exists(os.path.join(DATA_DIR, "train.jsonl")):
    say("preparing data:", DATA_RECIPE)
    subprocess.run(f"python -u prepare_data.py --recipe {DATA_RECIPE}", shell=True, check=True)
man = json.load(open(os.path.join(DATA_DIR, "manifest.json")))
os.makedirs("results", exist_ok=True)
shutil.copyfile(os.path.join(DATA_DIR, "manifest.json"), "results/speedtest_data_manifest.json")
used = {k: v for k, v in man["sources"].items() if v.get("train_rows", 0) > 0}
bad = [k for k in used if any(x in k.lower() for x in ("dbpedia", "bitext", "banking77", "clinc", "massive", "snips", "phrasebank", "sst5"))]
say("train rows", man["train_rows"], "sources used:", {k: v["train_rows"] for k, v in used.items()})
assert not bad, f"excluded / eval-set source in the data: {bad}"
assert man["train_rows"] >= 40 * 64

In [ ]:
sys.path.insert(0, WORK)
from ugc.config import load_config
LAST_OOM = False
NVSMI = {}      # per-gpu peak memory.used (MiB) of the current config, from an independent sampler

def sampler(stop):
    while not stop.is_set():
        try:
            out = subprocess.run("nvidia-smi --query-gpu=index,memory.used --format=csv,noheader,nounits", shell=True, capture_output=True, text=True, timeout=10).stdout
            for l in out.strip().splitlines():
                i, m = [x.strip() for x in l.split(",")]; NVSMI[int(i)] = max(NVSMI.get(int(i), 0), int(m))
        except Exception: pass
        stop.wait(2)

def run_streamed(cmd, env_extra, log_name, stall_min):
    global LAST_OOM
    LAST_OOM = False
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TORCH_NCCL_ASYNC_ERROR_HANDLING": "1", "TORCH_NCCL_DUMP_ON_TIMEOUT": "1", "NCCL_DEBUG": "WARN", **env_extra}
    path = os.path.join(LOG_DIR, f"{log_name}_{time.strftime('%H%M%S')}.log")
    say("RUN", cmd, "| log", path)
    p = subprocess.Popen(cmd, shell=True, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    q = queue.Queue()
    threading.Thread(target=lambda: ([q.put(l) for l in p.stdout], q.put(None)), daemon=True).start()
    last, stalled = time.time(), False
    with open(path, "a", buffering=1) as f:
        while True:
            try: line = q.get(timeout=10)
            except queue.Empty:
                if time.time() - last > stall_min * 60:
                    say(f"WATCHDOG: no output for {stall_min} min -> killing"); stalled = True
                    os.killpg(p.pid, signal.SIGTERM); time.sleep(20)
                    if p.poll() is None: os.killpg(p.pid, signal.SIGKILL)
                    break
                continue
            if line is None: break
            if re.search(r"OutOfMemoryError|CUDA out of memory", line): LAST_OOM = True
            last = time.time(); print(line, end="", flush=True); f.write(line)
    rc = p.wait(); say("EXIT", rc, "stalled" if stalled else "", "OOM" if LAST_OOM else "")
    return rc, stalled, path

def parse_log(path, nsmi):
    txt = open(path, errors="ignore").read()
    ranks = {}
    for m in re.finditer(r"rank(\d+)/\d+\] mem step=(\d+) peak_alloc_gb=([\d.]+) peak_reserved_gb=([\d.]+)", txt):
        r, st, a, rs = int(m.group(1)), int(m.group(2)), float(m.group(3)), float(m.group(4))
        d = ranks.setdefault(r, {"peak_alloc_gb": 0, "peak_reserved_gb": 0, "last_step": 0})
        d["peak_alloc_gb"] = max(d["peak_alloc_gb"], a); d["peak_reserved_gb"] = max(d["peak_reserved_gb"], rs); d["last_step"] = max(d["last_step"], st)
    sps = [float(x) for x in re.findall(r"'sec_per_step': '?([\d.]+)", txt)]
    return {"per_rank_torch": ranks, "nvidia_smi_peak_used_gb": {k: round(v / 1024, 2) for k, v in nsmi.items()},
            "sec_per_step_by_log_interval": sps, "sec_per_step_steady_mean_last3": round(sum(sps[-3:]) / len(sps[-3:]), 3) if sps else None}

def run_cfg(name):
    cfgp = f"configs/speedtest_{name}.yaml"
    sets = [f"model.init_from={INIT_LOCAL}", "data.val_files=null", f"train.time_budget_hours={max(0.05, min(CFG_MAX_HOURS, TOTAL_MAX_HOURS - elapsed_h() - 0.1)):.2f}"] + EXTRA_SET
    cfg = load_config(cfgp, sets)["train"]; out_dir = cfg["output_dir"]
    rec = {"config": name, "per_device_bs": cfg["per_device_train_batch_size"], "accum": cfg.get("gradient_accumulation_steps"),
           "grad_ckpt": cfg.get("gradient_checkpointing"), "max_oom_skips": cfg.get("max_oom_skips", 0), "t0_hours": round(elapsed_h(), 3)}
    say(f"===== {name}: bs/dev {rec['per_device_bs']} x accum {rec['accum']} x {NGPU} gpus, ckpt={rec['grad_ckpt']}, steps {cfg['max_steps']} =====")
    NVSMI.clear(); stop = threading.Event(); th = threading.Thread(target=sampler, args=(stop,), daemon=True); th.start()
    rc, stalled, logp = run_streamed(f"torchrun --standalone --nproc_per_node {NGPU} train.py --config {cfgp} --resume none --set " + " ".join(sets), {}, f"train_{name}", STALL_MINUTES)
    stop.set(); th.join(timeout=5)
    rec.update(rc=rc, stalled=stalled, oom=bool(LAST_OOM))
    st = json.load(open(os.path.join(out_dir, "STATUS.json"))) if os.path.exists(os.path.join(out_dir, "STATUS.json")) else {}
    rec["state"] = st.get("state", "no_status"); rec["global_step"] = st.get("global_step"); rec["train_loss"] = st.get("train_loss"); rec["session_seconds"] = st.get("session_seconds")
    rec.update(parse_log(logp, dict(NVSMI)))
    rec["wall_hours"] = round(elapsed_h() - rec["t0_hours"], 3)
    for c in glob.glob(os.path.join(out_dir, "checkpoint-*")) + glob.glob(os.path.join(out_dir, "final_model")):
        shutil.rmtree(c, ignore_errors=True)       # outputs of a speed test are worthless: keep STATUS/HEARTBEAT/logs only
    return rec

SUMMARY = []
def write_summary(note=""):
    json.dump({"session_hours": round(elapsed_h(), 3), "note": note, "configs": SUMMARY}, open("results/speedtest_summary.json", "w"), indent=2)
def reserved_max(rec):
    r = [d["peak_reserved_gb"] for d in rec.get("per_rank_torch", {}).values()]
    if len(r) >= NGPU: return max(r), "torch reserved, all ranks"
    s = list(rec.get("nvidia_smi_peak_used_gb", {}).values())
    return (max(s) if s else None), "nvidia-smi used (rank-1 torch line missing)"

# dist pre-flight (as in the pilot)
rc, st_, _ = run_streamed(f"torchrun --standalone --nproc_per_node {NGPU} scripts/dist_check.py", {}, "dist_check", 3)
assert rc == 0, "NCCL pre-flight failed: stop (no single-GPU fallback in a speed test)"

stop_reason = None
for name in CONFIGS_ORDER:
    if name == "T1_nockpt_b8":
        t2 = next((r for r in SUMMARY if r["config"] == "T2_nockpt_b4"), None)
        rm, how = reserved_max(t2) if t2 else (None, "T2 not run")
        if rm is None or rm >= GATE_T1_RESERVED_GB:
            say(f"T1 SKIPPED: T2 peak reserved = {rm} GB ({how}) is not < {GATE_T1_RESERVED_GB} GB")
            SUMMARY.append({"config": name, "skipped": True, "reason": f"T2 peak reserved {rm} GB ({how}) not < {GATE_T1_RESERVED_GB}"}); write_summary(); continue
        say(f"T1 gate OK: T2 peak reserved {rm} GB ({how}) < {GATE_T1_RESERVED_GB}")
    if elapsed_h() > TOTAL_MAX_HOURS - 0.2:
        stop_reason = f"session clock {elapsed_h():.2f} h past {TOTAL_MAX_HOURS - 0.2} h: {name} not started"; say("STOP:", stop_reason); break
    rec = run_cfg(name); SUMMARY.append(rec); write_summary()
    say(f"{name} done: state={rec['state']} oom={rec['oom']} rc={rec['rc']} sps={rec['sec_per_step_by_log_interval']} ranks={rec['per_rank_torch']} smi={rec['nvidia_smi_peak_used_gb']}")
    if rec["state"] != "complete" or rec["oom"] or rec["rc"] != 0:
        stop_reason = f"{name} failed (state={rec['state']} oom={rec['oom']} rc={rec['rc']}): stopping"; say("STOP:", stop_reason); break
write_summary(stop_reason or "all configs done")
print(json.dumps({"session_hours": round(elapsed_h(), 3), "stop_reason": stop_reason, "configs": SUMMARY}, indent=1))

In [ ]:
for p in glob.glob(os.path.join(DATA_DIR, "*.jsonl")): os.remove(p)     # keep the output small (data is re-creatable)
subprocess.run("du -sh runs data results /kaggle/working/logs 2>/dev/null", shell=True)
say("session hours used:", round(elapsed_h(), 3)); NB_LOG.close()